# 05 — Model Evaluation

**Goal:** Consolidate regression and classification metrics into a single
evaluation summary, saved to disk so the dashboard can load actual computed
metrics rather than hardcoded values.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import joblib
import json

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, f1_score

from src.ml.preprocessing import build_classification_dataset, get_classification_feature_target

# Regression test data + model
X_test_reg = pd.read_csv("../data/processed/X_test.csv")
y_test_reg = pd.read_csv("../data/processed/y_test.csv")["time_taken"]
regression_model = joblib.load("../models/regression_model.pkl")

# Classification: rebuild the same test split (same random_state=42)
unified_df = pd.read_csv("../data/processed/unified_dataset.csv")
classification_df = build_classification_dataset(unified_df)
X_clf, y_clf = get_classification_feature_target(classification_df)

from sklearn.model_selection import train_test_split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)

classification_model = joblib.load("../models/classification_model.pkl")

print("Regression test set:", X_test_reg.shape)
print("Classification test set:", X_test_clf.shape)
print("Both models loaded successfully.")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_3052\1545013504.py:20: DtypeWarning: Columns (0: found) have mixed types. Specify dtype option on import or set low_memory=False.
  unified_df = pd.read_csv("../data/processed/unified_dataset.csv")


Regression test set: (12948, 11)
Classification test set: (400, 6)
Both models loaded successfully.


In [2]:
# Regression metrics
reg_preds = regression_model.predict(X_test_reg)
reg_r2 = r2_score(y_test_reg, reg_preds)
reg_mae = mean_absolute_error(y_test_reg, reg_preds)
reg_rmse = np.sqrt(mean_squared_error(y_test_reg, reg_preds))

# Classification metrics
clf_preds = classification_model.predict(X_test_clf)
clf_accuracy = accuracy_score(y_test_clf, clf_preds)
clf_f1_macro = f1_score(y_test_clf, clf_preds, average="macro", zero_division=0)
clf_f1_weighted = f1_score(y_test_clf, clf_preds, average="weighted", zero_division=0)

print("=== Regression (Random Forest) ===")
print(f"R2:   {reg_r2:.4f}")
print(f"MAE:  {reg_mae:.6f}")
print(f"RMSE: {reg_rmse:.6f}")

print("\n=== Classification (Random Forest) ===")
print(f"Accuracy:    {clf_accuracy:.4f}")
print(f"Macro F1:    {clf_f1_macro:.4f}")
print(f"Weighted F1: {clf_f1_weighted:.4f}")

=== Regression (Random Forest) ===
R2:   0.9922
MAE:  0.002419
RMSE: 0.008600

=== Classification (Random Forest) ===
Accuracy:    0.8750
Macro F1:    0.5968
Weighted F1: 0.8757


In [3]:
import os
os.makedirs("../models", exist_ok=True)

evaluation_summary = {
    "regression": {
        "model": "Random Forest Regressor",
        "hyperparameters": {"n_estimators": 100, "max_depth": 12, "min_samples_leaf": 5},
        "r2": reg_r2,
        "mae": reg_mae,
        "rmse": reg_rmse,
        "test_set_size": len(y_test_reg),
    },
    "classification": {
        "model": "Random Forest Classifier",
        "hyperparameters": {"n_estimators": 200},
        "accuracy": clf_accuracy,
        "f1_macro": clf_f1_macro,
        "f1_weighted": clf_f1_weighted,
        "test_set_size": len(y_test_clf),
        "note": "selection_sort had 0 test instances (4 total, all in train); quick_sort had 3 test instances (9 total)."
    },
    "dataset": {
        "total_rows": len(unified_df),
        "sorting_rows": int((unified_df["domain"] == "sorting").sum()),
        "searching_rows": int((unified_df["domain"] == "searching").sum()),
        "unique_profiles": len(classification_df),
    }
}

with open("../models/evaluation_summary.json", "w") as f:
    json.dump(evaluation_summary, f, indent=2)

print("Saved evaluation_summary.json")
print(json.dumps(evaluation_summary, indent=2))

Saved evaluation_summary.json
{
  "regression": {
    "model": "Random Forest Regressor",
    "hyperparameters": {
      "n_estimators": 100,
      "max_depth": 12,
      "min_samples_leaf": 5
    },
    "r2": 0.992248916634641,
    "mae": 0.0024188120273771025,
    "rmse": 0.008600213707556356,
    "test_set_size": 12948
  },
  "classification": {
    "model": "Random Forest Classifier",
    "hyperparameters": {
      "n_estimators": 200
    },
    "accuracy": 0.875,
    "f1_macro": 0.5968169049211507,
    "f1_weighted": 0.8756853144162252,
    "test_set_size": 400,
    "note": "selection_sort had 0 test instances (4 total, all in train); quick_sort had 3 test instances (9 total)."
  },
  "dataset": {
    "total_rows": 64780,
    "sorting_rows": 34992,
    "searching_rows": 29788,
    "unique_profiles": 2000
  }
}


In [4]:
summary_table = pd.DataFrame([
    {"Task": "Regression (execution time)", "Metric": "R²", "Value": f"{reg_r2:.4f}"},
    {"Task": "Regression (execution time)", "Metric": "MAE", "Value": f"{reg_mae:.6f}s"},
    {"Task": "Regression (execution time)", "Metric": "RMSE", "Value": f"{reg_rmse:.6f}s"},
    {"Task": "Classification (best algorithm)", "Metric": "Accuracy", "Value": f"{clf_accuracy:.4f}"},
    {"Task": "Classification (best algorithm)", "Metric": "Macro F1", "Value": f"{clf_f1_macro:.4f}"},
    {"Task": "Classification (best algorithm)", "Metric": "Weighted F1", "Value": f"{clf_f1_weighted:.4f}"},
])
summary_table

,Task,Metric,Value
0,Regression (execution time),R²,0.9922
1,Regression (execution time),MAE,0.002419s
2,Regression (execution time),RMSE,0.008600s
3,Classification (best algorithm),Accuracy,0.8750
4,Classification (best algorithm),Macro F1,0.5968
5,Classification (best algorithm),Weighted F1,0.8757


## Evaluation Summary

| Task | Model | Key Metric | Value |
|---|---|---|---|
| Regression (execution time) | Random Forest Regressor | R² | 0.9922 |
| Regression (execution time) | Random Forest Regressor | MAE | 0.0024s |
| Classification (best algorithm) | Random Forest Classifier | Accuracy | 0.8750 |
| Classification (best algorithm) | Random Forest Classifier | Macro F1 | 0.5968 |

Both models are saved (`models/regression_model.pkl`, `models/classification_model.pkl`)
along with this consolidated summary (`models/evaluation_summary.json`), which the
interactive dashboard reads directly rather than hardcoding any values.

This evaluation sets up the central research question explored in the next phase:
**why is R²=0.99 achievable for time prediction, while classification F1 sits
at 0.60?** See notebook 06 (predictability gap analysis) for the full discussion.